In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="qwen3:4b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='067f56ff-8303-4be1-af88-e3c6ead9e3eb'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 665, 'prompt_tokens': 276, 'total_tokens': 941, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-137', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03894-347d-7fd2-afb8-b0bffab72cc8-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'call_2iuldrtx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 665, 'total_tokens': 941, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content=[{'type': 'text', 'text': 'Error execut

## Online MCP

In [8]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [9]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [10]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='736b8c41-00c7-436e-9618-966a9a805045'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 649, 'prompt_tokens': 134, 'total_tokens': 783, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-393', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03897-f24a-7120-94ad-20868fd0b1f2-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'current time'}, 'id': 'call_n4cswn42', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 134, 'output_tokens': 649, 'total_tokens': 783, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content=[{'type': 'text', 'text': 'Error executing tool search_web: Invalid API key: Unauthorized